In [1]:
# Author: Gergely Zahoranszky-Kohalmi, PhD
#
# Organization: National Center for Advancing Translational Sciences (NCATS/NIH)
#
# Email: gergely.zahoranszky-kohalmi@nih.gov


In [2]:
# Env: routesim

import requests
import json
import pandas as pd
import networkx as nx

import time

import os

import rdkit
from rdkit import Chem

import time

In [3]:
# Config Section
FNAME_IN_1 = '../data/input/SMILES.txt'
FNAME_IN_2 = '../data/input/retrostar_subset_tms.tsv'

#FNAME_IN_TMS = '../data/input/target_molecules.txt'

#TARGET_MOLECULE_INCHIKEY = 'HPTQHXKWSUVNNR-UHFFFAOYSA-N'


#SHORTEST_PATH = True

SEARCH_DEPTH = 5

SEARCH_TYPE = 'shortest_path'

TOP_X = 10

FNAME_OUT_DIR = '../data/output/routes/'

DIRECTORY_API_RESPONSE = "../data/output/api_json_files"

LEAVES_AS_SM = True

INCLUDE_AVAILABILITY_INFO = False

ANNOTATE_REACTIONS = False

GRAPH_BACKEND = 'memgraph'

INCLUDE_ROUTE_CANDIDATES = True

INCLUDE_COMBINATION_GRAPHS = True

SYNTHESIS_GRAPH_JSON = 'null'


# URLs

URL_ROUTE_SEARCH = "http://localhost:8002/syngps-app/api/v1/prediction/top_yield_routes"


RESULTS = {}





In [4]:
df = pd.read_csv (FNAME_IN_1, sep = '\t', header = None)
df.columns = ['smiles']
#print (df.head)

df2 = pd.read_csv (FNAME_IN_2, sep = '\t')
#print (df2.head)

df = pd.concat([df, df2])
print (df.head)


<bound method NDFrame.head of                                                 smiles
0    Cc1c(Cl)c2c(Cl)c(C)c1-c1c(-c3ccc(F)cc3)sc3ncnc...
1    CC(C)CCN[C@@H]1CCc2cc(O)c(N3CC(=O)NS3(=O)=O)c(...
2       CNC(=O)COc1cc(Cl)c(Cc2ccc(O)c(C(C)C)c2)c(Cl)c1
3    Cc1ncsc1-c1ccc([C@@H](CO)NC(=O)[C@H]2C[C@H](O)...
4    Cc1ncn(CC(=O)N2CCN(c3sc(C(F)(F)F)nc3-c3cnc(C(F...
..                                                 ...
195   CCOP(=O)(Cc1cc2cc(OC)c(OC)cc2nc1CC(N)C(=O)OC)OCC
196  CC(C)(C)OC(=O)NCc1ccc(C(=O)Nc2ccc(Cl)c(-c3cccc...
197  COc1ccc(N(c2ccc(OC)cc2)c2ccc(N(c3ccc(OC)cc3)c3...
198                     CC(=O)c1ccc(-c2ccc(Br)cc2F)cc1
199             CCOC(=O)c1ccc(N2C[C@H]3C[C@@H]2CN3)cc1

[372 rows x 1 columns]>


In [5]:
# Functions


def fetch_synthesis_routes(target_molecule, search_depth = 2, top_n = 3, search_type = "shortest_path", leaves_as_sm = True):
    """
        Search depths: maximal number of reactions steps to explore.
    """
    
    # Base URL for your API

    

    
    try:
        # Step 1: Make POST request to /api/test
        print("Fetching synthesis route ...")
        
        # Optional: Include data in the first request if needed




        synth_route_search_payload = {
            "target_molecule_inchikey": target_molecule,
            "reaction_steps": search_depth,
            "query_type": search_type,
            "leaves_as_sm": leaves_as_sm,
            "include_availability_info": INCLUDE_AVAILABILITY_INFO,
            "annotate_reactions": ANNOTATE_REACTIONS,
            "graph_backend": GRAPH_BACKEND,
            "top_n_routes": top_n,
            "include_route_candidates": INCLUDE_ROUTE_CANDIDATES,
            "include_combination_graphs": INCLUDE_COMBINATION_GRAPHS,
            "synthesis_graph_json": None,
            "inventory_source": "enamine"
        }
        # Note above new "inventory_source" parameter, which can be set to 'enamine', 'askcos', 'emolecules', or 'stock' (for the custom inventory file).



        # synth_route_search_payload = {
        #     "target_molecule_inchikey": target_molecule_inchikey,
        #     "reaction_steps": search_depth,
        #     "query_type": search_type,
        #     "leaves_as_sm": leaves_as_sm,
        #     "include_availability_info": False,
        #     "annotate_reactions": False,
        #     "graph_backend": "memgraph",
        #     "top_n_routes": top_n,
        #     "include_route_candidates": False,
        #     "include_combination_graphs": False
        # }
        
        response1 = requests.post(
            url = URL_ROUTE_SEARCH,
            json = synth_route_search_payload,
            headers={'Content-Type': 'application/json',
                     'accept': 'application/json'},
            verify=False
        )
        
        # Check if the request was successful
        #response1.raise_for_status()
        
        # Get JSON data from response
        received_data = response1.json()
        #print(f"Received data: {json.dumps(received_data, indent=2)}")

        return (received_data)
    
    except requests.exceptions.RequestException as e:
    
        print(f"Error occurred: {e}")

        return (None)


def save_json_to_file(data, filename=None, directory="api_json_files"):
    """
    Save JSON data to a local file with proper formatting.
    
    Args:
        data: Dictionary or JSON-serializable object
        filename: Optional custom filename. If None, auto-generates with timestamp
        directory: Directory to save the file (default: "api_json_files")
    
    Returns:
        str: Path to the saved file
    """
    # Create directory if it doesn't exist
    if not os.path.exists(directory):
        os.makedirs(directory)
    
    
    # Ensure filename has .json extension
    if not filename.endswith('.json'):
        filename += '.json'
    
    # Full file path
    filepath = os.path.join(directory, filename)
    
    # Save JSON with proper formatting
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    
    print(f"✓ JSON saved to: {filepath}")
    return filepath


def smiles2inchikey (smi):
    try:
        mol = Chem.MolFromSmiles(smi)
        ik = Chem.MolToInchiKey(mol)
        return (ik)

    except:
        print (f'[W] InChI-Key could not be generate for molecule: {smi} .')

    return ('IKERROR')

    

In [6]:
# Workflow

# Get Synthesis Route

df['inchikey'] = df.apply(lambda x: smiles2inchikey (x['smiles']), axis = 1)

tm_inchikeys = list(df['inchikey'])


idx = 1

time_start = time.time()

print (f'[*] Nr. of TMs: {len(tm_inchikeys)}')

for ik in tm_inchikeys:

    print (f'[*] Processing TM: {ik}, mol {idx}/{len(tm_inchikeys)}')



    idx += 1
    
    result = fetch_synthesis_routes (ik, search_depth = SEARCH_DEPTH, top_n = TOP_X, search_type = SEARCH_TYPE, leaves_as_sm = LEAVES_AS_SM)

    #print (result1)

    fname = ik + '_' + 'evidence_based_route.json'

    save_json_to_file(result, filename = fname, directory = FNAME_OUT_DIR)



time_end = time.time()

duration_sec = time_end - time_start

print (f'[*] Processing time (s): {duration_sec} .')

print ('[Done.]')




[*] Nr. of TMs: 372
[*] Processing TM: BOMNURVTAHAEBQ-KWIIHVIGSA-N, mol 1/372
Fetching synthesis route ...
✓ JSON saved to: ../data/output/routes/BOMNURVTAHAEBQ-KWIIHVIGSA-N_evidence_based_route.json
[*] Processing TM: DVFCRTGTEXUFIN-GFCCVEGCSA-N, mol 2/372
Fetching synthesis route ...
✓ JSON saved to: ../data/output/routes/DVFCRTGTEXUFIN-GFCCVEGCSA-N_evidence_based_route.json
[*] Processing TM: CQELSEDWYWTMDG-UHFFFAOYSA-N, mol 3/372
Fetching synthesis route ...
✓ JSON saved to: ../data/output/routes/CQELSEDWYWTMDG-UHFFFAOYSA-N_evidence_based_route.json
[*] Processing TM: DQRZNYPHOWVXPQ-VTUXAHAWSA-N, mol 4/372
Fetching synthesis route ...
✓ JSON saved to: ../data/output/routes/DQRZNYPHOWVXPQ-VTUXAHAWSA-N_evidence_based_route.json
[*] Processing TM: FDZFVXKCJKOSRC-CYBMUJFWSA-N, mol 5/372
Fetching synthesis route ...
✓ JSON saved to: ../data/output/routes/FDZFVXKCJKOSRC-CYBMUJFWSA-N_evidence_based_route.json
[*] Processing TM: AAZPIQPULVRHOW-SFKJMYEFSA-N, mol 6/372
Fetching synthesis rou

In [7]:
# Ref: https://www.freecodecamp.org/news/loading-a-json-file-in-python-how-to-read-and-parse-json/
# Ref: https://networkx.org/documentation/stable/reference/readwrite/generated/networkx.readwrite.json_graph.cytoscape_data.html#networkx.readwrite.json_graph.cytoscape_data
# Ref: https://networkx.org/documentation/stable/reference/readwrite/generated/networkx.readwrite.json_graph.cytoscape_graph.html
# Ref: https://groups.google.com/g/cytoscape-helpdesk/c/keumGM-bwz0
# Ref: https://networkx.org/documentation/networkx-1.9/reference/generated/networkx.readwrite.graphml.write_graphml.html
# Ref: https://github.com/cytoscape/copycat-layout/blob/master/notebooks/Copycat%20Automation%20Example.ipynb
# Ref: ChatGPT 4.0 [https://chat.openai.com]
# Ref: https://www.geeksforgeeks.org/python-map-function/
# Ref: https://stackoverflow.com/questions/24898797/check-if-key-exists-and-iterate-the-json-array-using-python
# Ref: https://htmlcolorcodes.com/colors/brick-red/
# Ref: https://stackoverflow.com/questions/32652149/combine-join-networkx-graphs
# Ref: https://networkx.org/documentation/stable/reference/classes/generated/networkx.Graph.copy.html
# Ref: https://www.w3schools.com/python/ref_requests_get.asp
# Ref: https://builtin.com/articles/timing-functions-python
# Ref: https://networkx.org/documentation/stable/reference/classes/generated/networkx.Graph.neighbors.html
# Ref: https://github.com/ncats/smartgraph_api_pub
# Ref: https://stackoverflow.com/questions/9733638/how-can-i-post-json-data-with-pythons-requests-library
# Ref: https://www.rdkit.org/docs/source/rdkit.Chem.rdinchi.html
#

